# Chatbot with Transformers

Build a multi-turn chatbot using **DialoGPT**, a GPT-2 model fine-tuned on Reddit conversations. We'll implement conversation history management and handle context window limits.

Key challenges in chatbot design:
- Maintaining **conversation history** across turns
- Managing **context window** limits (model has fixed max token length)
- Generating **coherent, non-repetitive** responses

## Setup

In [ ]:
!pip install transformers datasets sentence-transformers pillow -q

## Imports

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}")

## 1. Load DialoGPT

In [ ]:
MODEL = "microsoft/DialoGPT-small"  # use "medium" for better quality (slower)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)

print(f"Model loaded: {MODEL}")
print(f"Vocab size  : {tokenizer.vocab_size:,}")
print(f"Max length  : {tokenizer.model_max_length}")

## 2. Single-Turn Response

In [ ]:
def get_response(user_input, chat_history_ids=None):
    """Generate a response given user input and optional chat history."""
    # Encode user input + EOS token
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors="pt").to(device)

    # Append to history (or start fresh)
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Truncate if exceeding max length (keep recent context)
    max_len = 512
    if bot_input_ids.shape[-1] > max_len:
        bot_input_ids = bot_input_ids[:, -max_len:]

    # Generate
    with torch.no_grad():
        chat_history_ids = model.generate(
            bot_input_ids,
            max_new_tokens=60,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True, top_p=0.92, temperature=0.8,
            repetition_penalty=1.3,
        )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )
    return response, chat_history_ids

# Test single turn
response, _ = get_response("Hello! How are you?")
print(f"User : Hello! How are you?")
print(f"Bot  : {response}")

## 3. Multi-Turn Conversation with History

The model sees the full conversation history — each response is conditioned on everything said before.

In [ ]:
conversation = [
    "Hi! What's your name?",
    "What do you like to do for fun?",
    "Do you enjoy reading books?",
    "What kind of books do you like?",
    "That sounds interesting! Tell me more.",
]

history_ids = None
print("=== Simulated conversation ===\n")
for user_msg in conversation:
    response, history_ids = get_response(user_msg, history_ids)
    print(f"User : {user_msg}")
    print(f"Bot  : {response}")
    print(f"       (history length: {history_ids.shape[-1]} tokens)\n")